# FACTS - In-Class Example (variable line reactance)

**Original AMPL author:** Xingpeng Li - Associate Professor, Dept. of Electrical and Computer Engineering, University of Houston (UH), Houston, TX, USA (Senior Member, IEEE). Email: xli83@central.uh.edu

**Converted by:** Haoxiang Wan - PhD student of Dr. Xingpeng Li (AMPL -> Pyomo + Gurobi)

Two-bus system with FACTS-enabled variable reactance. Original AMPL used MINOS; Pyomo+Gurobi solves it with `NonConvex=2`.

In [1]:
from pyomo.environ import (
    ConcreteModel, Set, Param, Var, Objective, Constraint, SolverFactory,
    NonNegativeReals, minimize, value
)

# ---- Data: loaded from external file 'facts_data_case.txt' ----
import sys, pathlib
# Make the ampl_data parser importable (one level up from this notebook)
_pkg = pathlib.Path.cwd().parent
if str(_pkg) not in sys.path: sys.path.insert(0, str(_pkg))
from ampl_data import parse_ampl_data

_d = parse_ampl_data('facts_data_case.txt')

BUS            = _d['BUS']
GEN            = _d['GEN']
LOAD           = _d['LOAD']
BRANCH         = _d['BRANCH']
gen_Bus        = _d['gen_Bus']
gen_min_MW     = _d['gen_min']
gen_max_MW     = _d['gen_max']
gen_Cost       = _d['gen_Cost']
load_Bus       = _d['load_Bus']
load_Pd_MW     = _d['load_Pd']
branch_fromBus = _d['branch_fromBus']
branch_toBus   = _d['branch_toBus']
branch_x_orig  = _d['branch_x_orig']
branch_x_upPct = _d['branch_x_upPct']
branch_x_dnPct = _d['branch_x_dnPct']
branch_rate_MW = _d['branch_rate']

MVABase = 100.0




# Convert to per unit (mirrors AMPL "for" loops)
gen_min     = {g: gen_min_MW[g]/MVABase     for g in GEN}
gen_max     = {g: gen_max_MW[g]/MVABase     for g in GEN}
load_Pd     = {d: load_Pd_MW[d]/MVABase     for d in LOAD}
branch_rate = {k: branch_rate_MW[k]/MVABase for k in BRANCH}

m = ConcreteModel()
m.BUS    = Set(initialize=BUS,    ordered=True)
m.GEN    = Set(initialize=GEN,    ordered=True)
m.LOAD   = Set(initialize=LOAD,   ordered=True)
m.BRANCH = Set(initialize=BRANCH, ordered=True)

m.gen_Bus  = Param(m.GEN, initialize=gen_Bus)
m.gen_Cost = Param(m.GEN, initialize=gen_Cost)
m.gen_min  = Param(m.GEN, initialize=gen_min)
m.gen_max  = Param(m.GEN, initialize=gen_max)

m.load_Bus = Param(m.LOAD, initialize=load_Bus)
m.load_Pd  = Param(m.LOAD, initialize=load_Pd)

m.branch_fromBus = Param(m.BRANCH, initialize=branch_fromBus)
m.branch_toBus   = Param(m.BRANCH, initialize=branch_toBus)
m.branch_x_orig  = Param(m.BRANCH, initialize=branch_x_orig)
m.branch_x_upPct = Param(m.BRANCH, initialize=branch_x_upPct)
m.branch_x_dnPct = Param(m.BRANCH, initialize=branch_x_dnPct)
m.branch_rate    = Param(m.BRANCH, initialize=branch_rate)

m.pg = Var(m.GEN, domain=NonNegativeReals,
           bounds=lambda mm, g: (mm.gen_min[g], mm.gen_max[g]))
m.pk = Var(m.BRANCH,
           bounds=lambda mm, k: (-mm.branch_rate[k], mm.branch_rate[k]))
m.theta = Var(m.BUS)
m.branch_x = Var(m.BRANCH,
                 bounds=lambda mm, k: (mm.branch_x_orig[k]*(1 - mm.branch_x_dnPct[k]),
                                       mm.branch_x_orig[k]*(1 + mm.branch_x_upPct[k])))

# ---- Objective ----
# Note: FACTS is non-convex and has a continuous family of optimal solutions
# (multiple branch_x configurations yield the same generation cost).  The
# tie-breaker term `- 1e-3 * sum(branch_x)` below is COMMENTED OUT by
# default.  Uncomment it to make Gurobi converge to the same branch_x
# point as AMPL/MINOS (and the slides).  The optimal objective value is
# UNCHANGED either way, because multiple optimal solutions exist.
m.obj = Objective(
    rule=lambda mm: sum(mm.pg[g]*mm.gen_Cost[g]*MVABase for g in mm.GEN)
                    # - 1e-3 * sum(mm.branch_x[k] for k in mm.BRANCH)   # tie-breaker (disabled)
                    ,
    sense=minimize
)

def nodal_rule(mm, n):
    gen_in  = sum(mm.pg[g]   for g in mm.GEN    if mm.gen_Bus[g]       == n)
    out_flw = sum(mm.pk[k]   for k in mm.BRANCH if mm.branch_fromBus[k] == n)
    in_flw  = sum(mm.pk[k]   for k in mm.BRANCH if mm.branch_toBus[k]   == n)
    load    = sum(mm.load_Pd[d] for d in mm.LOAD if mm.load_Bus[d]    == n)
    return gen_in - out_flw + in_flw == load
m.nodalPB = Constraint(m.BUS, rule=nodal_rule)

# Bilinear flow constraint (variable reactance * angle difference)
m.lineFlow = Constraint(m.BRANCH,
    rule=lambda mm, k: mm.pk[k]*mm.branch_x[k]
                      == (mm.theta[mm.branch_fromBus[k]] - mm.theta[mm.branch_toBus[k]]))

# Reference bus
m.theta[1].fix(0)

model = m

In [2]:
# Bilinear constraint (branch_x * pk = angle diff) -> needs non-convex setting in Gurobi.
solver = SolverFactory('gurobi')
solver.options['NonConvex'] = 2
solver.options['MIPGap']   = 0.0
solver.options['TimeLimit'] = 90
results = solver.solve(model, tee=True)
print(results.solver.status, results.solver.termination_condition)

m = model

# ---- AMPL-style output ----
print("pg[g]*MVABase [*] :=")
for g in m.GEN:
    print(f"{g} {value(m.pg[g])*MVABase:.4g}")
print(";")

print()
print("pk[k]*MVABase [*] :=")
for k in m.BRANCH:
    print(f"{k} {value(m.pk[k])*MVABase:.4g}")
print(";")

print()
print(": branch_x branch_x_orig branch_x_upPct branch_x_dnPct :=")
for k in m.BRANCH:
    print(f"{k} {value(m.branch_x[k]):.4g} {value(m.branch_x_orig[k]):.4g} "
          f"{value(m.branch_x_upPct[k]):.4g} {value(m.branch_x_dnPct[k]):.4g}")
print(";")

print()
print("branch_x[k]/0.01 [*] :=")
for k in m.BRANCH:
    print(f"{k} {value(m.branch_x[k])/0.01:.4g}")
print(";")


Read LP format model from file C:\Users\hwan6\AppData\Local\Temp\tmppa3ayhvf.pyomo.lp
Reading time = 0.00 seconds
x1: 2 rows, 9 columns, 8 nonzeros
Set parameter NonConvex to value 2
Set parameter MIPGap to value 0
Set parameter TimeLimit to value 90
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-12700, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 20 logical processors, using up to 20 threads

Non-default parameters:
TimeLimit  90
MIPGap  0
NonConvex  2

Optimize a model with 2 rows, 9 columns and 8 nonzeros
Model fingerprint: 0xaabd3263
Model has 3 quadratic constraints
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  QMatrix range    [1e+00, 1e+00]
  QLMatrix range   [1e+00, 1e+00]
  Objective range  [1e+03, 3e+03]
  Bounds range     [5e-03, 2e+00]
  RHS range        [2e+00, 2e+00]

Continuous model is non-convex -- solving as a MIP

Presolve removed 1 rows and 1 columns
Presol

Reading time = 0.00 seconds
x1: 2 rows, 9 columns, 8 nonzeros
Set parameter NonConvex to value 2
Set parameter MIPGap to value 0
Set parameter TimeLimit to value 90
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-12700, instruction set [SSE2|AVX|AVX2]


Thread count: 12 physical cores, 20 logical processors, using up to 20 threads

Non-default parameters:


TimeLimit  90
MIPGap  0
NonConvex  2


Optimize a model with 2 rows, 9 columns and 8 nonzeros
Model fingerprint: 0xaabd3263
Model has 3 quadratic constraints
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  QMatrix range    [1e+00, 1e+00]
  QLMatrix range   [1e+00, 1e+00]
  Objective range  [1e+03, 3e+03]
  Bounds range     [5e-03, 2e+00]


...


Presolve removed 1 rows and 1 columns


Presolve time: 0.00s
Presolved: 13 rows, 9 columns, 40 nonzeros
Presolved model has 3 bilinear constraint(s)


Variable types: 9 continuous, 0 integer (0 binary)
Found heuristic solution: objective 2900.0000000



Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.00 work units)
Thread count was 20 (of 20 available processors)

Solution count 1: 2900 

Optimal solution found (tolerance 0.00e+00)
Best objective 2.900000000000e+03, best bound 2.900000000000e+03, gap 0.0000%


ok optimal
pg[g]*MVABase [*] :=
1 80
2 70
;

pk[k]*MVABase [*] :=
1 50
2 20
3 10
;

: branch_x branch_x_orig branch_x_upPct branch_x_dnPct :=
1 0.005401 0.01 0.5 0.5
2 0.0135 0.02 0.5 0.5
3 0.027 0.02 0.5 0.5
;

branch_x[k]/0.01 [*] :=
1 0.5401
2 1.35
3 2.7
;
